# Personal Budget Assistant -- Agent Demo

CSE476 CA1 Project 1. Runs the same `run_agent` loop from `agent.py` on 3 example goals and prints the full multi-step trace (tool calls, arguments, and results) for each one -- this is the proof the agent plans and acts rather than just answering once.


In [1]:
import os
from getpass import getpass
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Groq API key: ")

from agent import run_agent
from memory import SessionMemory

memory = SessionMemory()


## Goal 1: set a budget and log two expenses

Expect: two `add_expense` tool calls, then a plain-text confirmation.

In [2]:
run_agent(
    "My monthly budget is 20000 rupees. Log a 3000 expense for rent, category housing, "
    "and a 450 expense for groceries, category food.",
    memory,
)


[step 1] called add_expense({"amount":3000,"category":"housing","item":"rent"}) -> {'logged': {'item': 'rent', 'amount': 3000.0, 'category': 'housing'}, 'total_spent': 3000.0, 'expense_count': 1}
[step 2] called add_expense({"amount":450,"category":"food","item":"groceries"}) -> {'logged': {'item': 'groceries', 'amount': 450.0, 'category': 'food'}, 'total_spent': 3450.0, 'expense_count': 2}
[step 3] final answer:
Your monthly budget is set to **20,000 rupees**.

I’ve logged the two expenses:
- **Rent** (housing): 3,000 rupees  
- **Groceries** (food): 450 rupees  

Feel free to ask about your remaining budget, category totals, or anything else!



'Your monthly budget is set to **20,000\u202frupees**.\n\nI’ve logged the two expenses:\n- **Rent** (housing): 3,000\u202frupees  \n- **Groceries** (food): 450\u202frupees  \n\nFeel free to ask about your remaining budget, category totals, or anything else!'

## Goal 2: category summary

Expect: a `get_summary` tool call filtered to `food`, then an answer built from the tool result -- not a guess.

In [3]:
run_agent("How much have I spent on food so far?", memory)


[step 1] called get_summary({"category":"food"}) -> {'category': 'food', 'total': 450.0, 'count': 1}
[step 2] final answer:
You’ve spent **450 rupees** on food so far. This comes from the grocery expense you logged earlier. Let me know if you’d like to see totals for other categories or your overall remaining budget!



'You’ve spent **450\u202frupees** on food so far. This comes from the grocery expense you logged earlier. Let me know if you’d like to see totals for other categories or your overall remaining budget!'

## Goal 3: affordability decision (the agentic test)

Expect: a `get_summary` tool call, then the model subtracting the running total from the stated 20000 budget and deciding -- this is the "can I afford X" case the assignment calls out explicitly.

In [4]:
run_agent("Can I afford a 2000 rupee weekend trip this month?", memory)


[step 1] called get_summary({"category":null}) -> {'total_spent': 3450.0, 'by_category': {'housing': 3000.0, 'food': 450.0}, 'expense_count': 2}
[step 2] final answer:
Here’s the current picture:

| Item | Amount (₹) |
|------|------------|
| **Total spent so far** | **3,450** |
| **Monthly budget** | **20,000** |
| **Remaining budget** | **16,550** |

A weekend trip that costs **₹2,000** would bring your total spending to **₹5,450**, which is well under the remaining **₹16,550**.  

**Yes, you can afford the trip** and you’d still have **₹14,550** left for the rest of the month. Let me know if you’d like to log the trip expense or check any other category totals!



'Here’s the current picture:\n\n| Item | Amount (₹) |\n|------|------------|\n| **Total spent so far** | **3,450** |\n| **Monthly budget** | **20,000** |\n| **Remaining budget** | **16,550** |\n\nA weekend trip that costs **₹2,000** would bring your total spending to **₹5,450**, which is well under the remaining **₹16,550**.  \n\n**Yes, you can afford the trip** and you’d still have **₹14,550** left for the rest of the month. Let me know if you’d like to log the trip expense or check any other category totals!'

## Full conversation memory (proof it remembers earlier turns)

In [5]:
for m in memory.history:
    print(m["role"], "->", m["content"])


user -> My monthly budget is 20000 rupees. Log a 3000 expense for rent, category housing, and a 450 expense for groceries, category food.
assistant -> Your monthly budget is set to **20,000 rupees**.

I’ve logged the two expenses:
- **Rent** (housing): 3,000 rupees  
- **Groceries** (food): 450 rupees  

Feel free to ask about your remaining budget, category totals, or anything else!
user -> How much have I spent on food so far?
assistant -> You’ve spent **450 rupees** on food so far. This comes from the grocery expense you logged earlier. Let me know if you’d like to see totals for other categories or your overall remaining budget!
user -> Can I afford a 2000 rupee weekend trip this month?
assistant -> Here’s the current picture:

| Item | Amount (₹) |
|------|------------|
| **Total spent so far** | **3,450** |
| **Monthly budget** | **20,000** |
| **Remaining budget** | **16,550** |

A weekend trip that costs **₹2,000** would bring your total spending to **₹5,450**, which is well un